# Tahapan Pipeline Cleaning & Standarisasi Job Dataset

Notebook ini menggabungkan seluruh proses cleaning dari tiga tahap utama menjadi satu pipeline yang berurutan:

| # | Tahap | Deskripsi |
|---|-------|-----------|
| 1 | **Cleaning & Merge Job Dataset** | Baca, gabungkan, dan bersihkan data scraping (data untuk FD) serta data Kaggle (Data untuk AI) |
| 2 | **Cleaning Kualifikasi via Groq API** | Perkaya kolom `kualifikasi` menggunakan LLM (Groq / LLaMA) |
| 3 | **Cleaning & Standarisasi Kolom `skill`** | Bersihkan noise dan standarisasi token skill menggunakan kamus IT |

---
## Import Library

In [1]:
import os
import glob
import re
import time
import getpass
import pandas as pd
from groq import Groq
from IPython.display import display

print('✅ Semua library berhasil diimport')

✅ Semua library berhasil diimport


---
# BAGIAN 1 — Cleaning & Merge Job Dataset

Bagian ini membaca semua file dataset dari dua sumber:
- **Dataset Scraping** (folder `dataset/`) : digunakan sebagai **Full Data (FD)**
- **Dataset Kaggle** (folder `add_dataset_job/`) : digunakan sebagai data AI

Keduanya kemudian digabungkan, dibersihkan, dan dipersiapkan untuk tahap selanjutnya.

## 1.1 Baca DataSet

### 1.1A. Baca Dataset Scraping (FD)

Semua file `.xlsx` di dalam folder `dataset/` dibaca secara otomatis menggunakan `glob`.
Fungsi `glob.glob()` mencari semua file yang cocok dengan pola path yang diberikan,
sehingga tidak perlu mendaftarkan nama file satu per satu.

In [2]:
# Tentukan folder dataset scraping
FOLDER_SCRAPING = "dataset"

all_files_scraping = glob.glob(os.path.join(FOLDER_SCRAPING, "*.xlsx"))
print(f"File xlsx scraping ditemukan ({len(all_files_scraping)}):")
for f in all_files_scraping:
    print(" -", os.path.basename(f))

File xlsx scraping ditemukan (24):
 - AI_Engineer.xlsx
 - Backend_Developer.xlsx
 - Cloud_Engineer.xlsx
 - Cyber_Security.xlsx
 - Database_Administrator.xlsx
 - Data_Analyst.xlsx
 - Data_Engineer.xlsx
 - Data_science.xlsx
 - DevOps_Engineer.xlsx
 - Frontend_Developer.xlsx
 - Fullstack_Developer.xlsx
 - IoT_Engineer.xlsx
 - IT_Consultant.xlsx
 - IT_Support.xlsx
 - Mobile_Developer.xlsx
 - Network_Engineer.xlsx
 - Product_Manager.xlsx
 - Project_Manager_IT.xlsx
 - Software Developer .xlsx
 - Software_Tester.xlsx
 - Staff_IT.xlsx
 - System_Administrator.xlsx
 - Technical_Support.xlsx
 - Web_Developer.xlsx


### 1.1B. Baca Dataset Kaggle (AI)

File-file Kaggle dibaca satu per satu dari folder `add_dataset_job/`.
Setiap DataFrame disimpan ke dalam dictionary `dataframes_kaggle` menggunakan nama file sebagai key,
sehingga mudah diakses kembali saat proses mapping kolom.

In [3]:
# Konfigurasi folder dataset Kaggle
FOLDER_KAGGLE = 'add_dataset_job'

file_names_kaggle = [
    'IT_Job_Roles_Skills.xlsx',
    'Job opportunities.xlsx',
    'job_titles_and_descriptions.xlsx',
    'jobs.xlsx',
    'JobsDatasetProcessed.xlsx',
    'postings.xlsx',
    'Scrapped_data.xlsx'
]

# Dictionary untuk menampung semua dataframe Kaggle
dataframes_kaggle = {}

for file in file_names_kaggle:
    file_path = os.path.join(FOLDER_KAGGLE, file)
    if os.path.exists(file_path):
        df = pd.read_excel(file_path)
        dataframes_kaggle[file] = df
        print(f"{'='*55}")
        print(f"Membaca Dataset: {file}")
        print(f"{'='*55}")
        df.info()
        print("\n>>> PREVIEW DATA (HEAD):")
        display(df.head())
        print("\n")
    else:
        print(f"[ERROR] File tidak ditemukan: {file_path}")
        print("-" * 55)

Membaca Dataset: IT_Job_Roles_Skills.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Job Title        493 non-null    object
 1   Job Description  493 non-null    object
 2   Skills           493 non-null    object
 3   Certifications   493 non-null    object
dtypes: object(4)
memory usage: 15.5+ KB

>>> PREVIEW DATA (HEAD):


,Job Title,Job Description,Skills,Certifications
0,.NET Developer,Develops applications using .NET framework tec...,".NET, C#, ASP.NET, SQL Server, Web API, MVC, E...","Microsoft Certified: .NET Developer, MCP: Prog..."
1,2D ARTIST/ANIMATOR,Creates 2D animations for various media.,"2D Animation, Drawing, Illustration, Animation...","Toon Boom Certified Artist, Autodesk SketchBoo..."
2,COMPUTER GRAPHICS ANIMATOR,Creates animations using computer graphics sof...,"3D Animation, 2D Animation, Motion Graphics, A...","Adobe Creative Suite Certifications, Maya Cert..."
3,3D ARTIST/ANIMATOR,Creates 3D animations for various media.,"3D Animation, Modeling, Texturing, Rigging","Autodesk Maya Certified Professional, Blender..."
4,TECHNICAL ACCOUNT MANAGER,Manages relationships with technical customers...,"Account Management, Technical Support, Custome...","AWS Certified Solutions Architect - Associate,..."




Membaca Dataset: Job opportunities.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364 entries, 0 to 363
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Job Title         364 non-null    object        
 1   Job Description   364 non-null    object        
 2   Required Skills   364 non-null    object        
 3   Salary Range      364 non-null    object        
 4   Location          364 non-null    object        
 5   Company           364 non-null    object        
 6   Experience Level  364 non-null    object        
 7   Industry          364 non-null    object        
 8   Job Type          364 non-null    object        
 9   Date Posted       364 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(9)
memory usage: 28.6+ KB

>>> PREVIEW DATA (HEAD):


,Job Title,Job Description,Required Skills,Salary Range,Location,Company,Experience Level,Industry,Job Type,Date Posted
0,Software Engineer,Develop software,"Java, Python","£40,000 - £60,000",London,ABC Tech,Entry-Level,Technology,Full-Time,2023-01-05
1,Data Analyst,Analyze data,"SQL, Excel","£35,000 - £50,000",Manchester,XYZ Analytics,Junior,Analytics,Full-Time,2023-02-10
2,Network Engineer,Maintain networks,"Cisco, WAN","£45,000 - £70,000",Birmingham,Network Solutions,Mid-Level,Networking,Full-Time,2023-03-15
3,Cloud Architect,Design cloud,"AWS, Azure","£60,000 - £90,000",Edinburgh,Cloud Innovators,Senior,Cloud Computing,Full-Time,2023-04-20
4,Cybersecurity Analyst,Protect data,Cybersecurity,"£50,000 - £80,000",Glasgow,SecureGuard,Mid-Level,Security,Full-Time,2023-05-25




Membaca Dataset: job_titles_and_descriptions.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289 entries, 0 to 288
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Job Title        289 non-null    object
 1   Skills           289 non-null    object
 2   Job Description  289 non-null    object
dtypes: object(3)
memory usage: 6.9+ KB

>>> PREVIEW DATA (HEAD):


...

,ID,Query,Job Title,Description,IT Skills,Soft Skills,Education,Experience,Token Usage
0,9747,Network Architect,Software Test Specialist I,Job Summary: This position tests configuration...,"### ****, Software Testing, Quality Assurance,...","### ****, Communication, Collaboration, Custom...","### **Education-Related Skills:**, Software Te...","### **Experience-Related Skills:**, Batch Test...",918
1,9972,Network Architect,Infrastructure Architect,Symantec Corporation is the global leader in c...,"###, BGP, OSPF, Carrier-grade router architect...","###, Leadership, Communication, Problem-solvin...",NaN,NaN,920
2,1369,Data Engineer,Data Engineer,"Los Gatos, California Data Engineering and Inf...","###, Data Engineering, Data Structures, Data P...","###, Passion for Learning, Autonomy, Freedom &...",NaN,NaN,914
3,339,Data Scientist,"Consultant, Reporting & Data Analysis","Consultant, Reporting & Data Analysis \- (1800...","###, Data modeling, Data analysis, Data consol...","###, Communication, Client-facing consulting, ...","### Education-Related Skills:, Bachelor's degr...","### Experience-Related Skills:, Travel industr...",989
4,476,Data Analyst,Data Analyst,"alliantgroup, LP is currently experiencing exp...","###, Data processing, Data entry, Quality assu...","###, Attention to detail, Accuracy, Ability to...",NaN,NaN,396




Membaca Dataset: postings.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6025 entries, 0 to 6024
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   job_title       6025 non-null   object        
 1   company         6025 non-null   object        
 2   job_location    6025 non-null   object        
 3   job_link        6025 non-null   object        
 4   first_seen      6025 non-null   datetime64[ns]
 5   search_city     6025 non-null   object        
 6   search_country  6025 non-null   object        
 7   job level       6025 non-null   object        
 8   job_type        6025 non-null   object        
 9   job_summary     5665 non-null   object        
 10  job_skills      4960 non-null   object        
dtypes: datetime64[ns](1), object(10)
memory usage: 517.9+ KB

>>> PREVIEW DATA (HEAD):


,job_title,company,job_location,job_link,first_seen,search_city,search_country,job level,job_type,job_summary,job_skills
0,Data Engineer 2,Cook Medical,"Bloomington, IN",https://www.linkedin.com/jobs/view/data-engine...,2023-12-17,Bloomington,United States,Mid senior,Onsite,"Overview\nThe Data Engineer develops, implemen...","Azure, SQL, NoSQL, SQL Server, Oracle, MongoDB..."
1,Staff Data Engineer,Recruiting from Scratch,"Bloomington, IN",https://www.linkedin.com/jobs/view/staff-data-...,2023-12-17,Bloomington,United States,Mid senior,Onsite,This is for a client of Recruiting from Scratc...,"Python, Snowflake, Airflow, Kubernetes, Docker..."
2,"Senior Data Engineer, Public Company",Recruiting from Scratch,"Bloomington, IN",https://www.linkedin.com/jobs/view/senior-data...,2023-12-17,Bloomington,United States,Mid senior,Onsite,This is for a client of Recruiting from Scratc...,"Python, SQL, Snowflake, Airflow, Kubernetes, D..."
3,"Senior Data Engineer, Public Company",Recruiting from Scratch,"Bloomington, IN",https://www.linkedin.com/jobs/view/senior-data...,2023-12-17,Bloomington,United States,Mid senior,Onsite,This is for a client of Recruiting from Scratc...,"TDD, Automation, Continuous delivery, Data eng..."
4,"Senior Systems Engineer, Azure Data Platform",Cook Medical,"Bloomington, IN",https://www.linkedin.com/jobs/view/senior-syst...,2023-12-17,Bloomington,United States,Mid senior,Hybrid,Overview\nWe are seeking a talented Azure Clou...,NaN




Membaca Dataset: Scrapped_data.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2250 entries, 0 to 2249
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Job Name          2249 non-null   object 
 1   Company Name      2250 non-null   object 
 2   JD                2245 non-null   object 
 3   Skills            2250 non-null   object 
 4   Date Posted       2250 non-null   object 
 5   YOE               2250 non-null   object 
 6   Location          2179 non-null   object 
 7   Website           1954 non-null   object 
 8   Job Function:     2250 non-null   object 
 9   Industry:         2250 non-null   object 
 10  Specialization:   2250 non-null   object 
 11  Qualification:    2250 non-null   object 
 12  Hiring Location:  6 non-null      object 
 13  Role:             87 non-null     object 
 14  Vacancies:        12 non-null     float64
dtypes: float64(1), object(14)
memory usage: 263.8+ KB



,Job Name,Company Name,JD,Skills,Date Posted,YOE,Location,Website,Job Function:,Industry:,Specialization:,Qualification:,Hiring Location:,Role:,Vacancies:
0,Python Engineer,east india securities ltd.,job_description 2 years of experience worki...,"python,hadoop,machinelearning",Posted 5 days ago,2 - 5 yrs,Kolkata,http://www.eisec.com/,IT Software : Software Products & Services,"Petroleum/Oil and Gas/Power, Construction/Cem...",Software Engineer,Any Graduate,NaN,NaN,NaN
1,PYTHON DEVELOPER,DREAMAJAX TECHNOLOGIES,"PYTHON DEVELOPER B.E / B.Tech / MCA / M.Sc., o...","python,django,api,sql,nosql",Posted 6 days ago,4 - 7 yrs,Bengaluru / Bangalore,NaN,IT Software : Software Products & Services,"Telecom, IT-Hardware/Networking",Software Engineer,MCA/ PGDCA_x000D_\n _x000D_...,NaN,NaN,NaN
2,Python Developer,InnOvator Web Solutions Pvt.Ltd.,Job Category: DevelopmentJob Type: Full TimeJo...,"rest,python,django,webdeveloper,mysql,api",Posted 6 days ago,5 - 8 yrs,Mumbai,http://www.innovatorwebsolutions.com,IT Software : Software Products & Services,"IT-Hardware/Networking, Telecom",Software Engineer,Any Graduate,NaN,NaN,NaN
3,Python - Odoo,AxisTechnolabs,Responsibilities : We are looking for Freshers...,"python,django,itskills,html5,api,jquery",Posted 6 days ago,0 - 1 yrs,Ahmedabad,http://www.axistechnolabs.com,IT Software : Software Products & Services,"IT-Hardware/Networking, Telecom",Software Engineer,MCA/ PGDCA_x000D_\n _x000D_...,NaN,NaN,NaN
4,Python Developer,pearl global solutions,"Python Developer Django Job Type : Full time,...","python,database,django,teamplayer,sql",Posted 6 days ago,4 - 7 yrs,Cochin/ Kochi/ Ernakulam,https://pearlglobalsolutions.com/,IT Software : Software Products & Services,"IT-Hardware/Networking, Telecom",Software Engineer,Any Graduate,NaN,NaN,NaN


## 1.2 Merge Dataset

### 1.2A. Merge Dataset Scraping → `df_scraping`

Semua file scraping yang sudah dibaca digabungkan menjadi satu DataFrame menggunakan `pd.concat()`.
Parameter `ignore_index=True` digunakan agar index di-reset dari 0 secara berurutan
dan tidak ada index duplikat dari masing-masing file.

In [4]:
df_list_scraping = []
for filepath in all_files_scraping:
    _df = pd.read_excel(filepath)
    df_list_scraping.append(_df)
    print(f"Loaded: {os.path.basename(filepath)} -> {_df.shape[0]} rows, {_df.shape[1]} columns")

df_scraping = pd.concat(df_list_scraping, ignore_index=True)
print(f"\n✅ Total setelah digabung (Scraping): {df_scraping.shape[0]} baris, {df_scraping.shape[1]} kolom")
df_scraping.head()

Loaded: AI_Engineer.xlsx -> 303 rows, 13 columns
Loaded: Backend_Developer.xlsx -> 562 rows, 13 columns
Loaded: Cloud_Engineer.xlsx -> 324 rows, 13 columns
Loaded: Cyber_Security.xlsx -> 235 rows, 13 columns
Loaded: Database_Administrator.xlsx -> 509 rows, 13 columns
Loaded: Data_Analyst.xlsx -> 170 rows, 13 columns
Loaded: Data_Engineer.xlsx -> 170 rows, 13 columns
Loaded: Data_science.xlsx -> 730 rows, 13 columns
Loaded: DevOps_Engineer.xlsx -> 645 rows, 13 columns
Loaded: Frontend_Developer.xlsx -> 728 rows, 13 columns
Loaded: Fullstack_Developer.xlsx -> 788 rows, 13 columns
Loaded: IoT_Engineer.xlsx -> 430 rows, 13 columns
Loaded: IT_Consultant.xlsx -> 551 rows, 13 columns
Loaded: IT_Support.xlsx -> 612 rows, 13 columns
Loaded: Mobile_Developer.xlsx -> 24 rows, 13 columns
Loaded: Network_Engineer.xlsx -> 428 rows, 13 columns
Loaded: Product_Manager.xlsx -> 27 rows, 13 columns
Loaded: Project_Manager_IT.xlsx -> 611 rows, 13 columns
Loaded: Software Developer .xlsx -> 698 rows, 13 co

,tautan,posisi,perusahaan,lokasi,gaji,jenis_pekerjaan,kategori,pendidikan,pengalaman,gender,usia,skill,kualifikasi
0,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > AI Engineer,Minimal Sarjana (S1),5 - 10 tahun pengalaman,NaN,NaN,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr..."
1,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > AI Engineer,Minimal Sarjana (S1),5 - 10 tahun pengalaman,NaN,NaN,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr..."
2,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > AI Engineer,Minimal Sarjana (S1),5 - 10 tahun pengalaman,NaN,NaN,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr..."
3,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > AI Engineer,Minimal Sarjana (S1),5 - 10 tahun pengalaman,NaN,NaN,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr..."
4,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > AI Engineer,Minimal Sarjana (S1),5 - 10 tahun pengalaman,NaN,NaN,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr..."


### 1.2B. Merge Dataset Kaggle → `df_kaggle`

Setiap file Kaggle memiliki nama kolom yang berbeda-beda.
`column_mapping` mendefinisikan mapping dari nama kolom asli ke nama kolom standar yang seragam.
Kolom yang tidak tersedia di suatu file akan diisi `pd.NA` secara otomatis agar struktur tetap konsisten
untuk semua file yang digabungkan.

In [5]:
# Mapping kolom dari masing-masing file Kaggle ke kolom standar
column_mapping = {
    'IT_Job_Roles_Skills.xlsx': {
        'Job Title': 'posisi',
        'skill': 'skill',
        'Job Description': 'kualifikasi'
    },
    'Job opportunities.xlsx': {
        'Job Title': 'posisi',
        'Required Skills': 'skill',
        'Experience Level': 'pengalaman',
        'Job Description': 'kualifikasi'
    },
    'job_titles_and_descriptions.xlsx': {
        'Job Title': 'posisi',
        'Skills': 'skill',
        'Job Description': 'kualifikasi'
    },
    'jobs.xlsx': {
        'Job_title': 'posisi',
        'skills_required': 'skill'
    },
    'JobsDatasetProcessed.xlsx': {
        'Job Title': 'posisi',
        'IT Skills': 'skill',
        'Education': 'pendidikan',
        'Experience': 'pengalaman',
        'Description': 'kualifikasi'
    },
    'postings.xlsx': {
        'job_title': 'posisi',
        'job_skills': 'skill',
        'job level': 'pengalaman',
        'job_summary': 'kualifikasi'
    },
    'Scrapped_data.xlsx': {
        'Job Name': 'posisi',
        'Skills': 'skill',
        'YOE': 'pengalaman',
        'JD': 'kualifikasi'
    }
}

# Target kolom akhir untuk AI
target_columns_ai = ['posisi', 'pendidikan', 'pengalaman', 'gender', 'usia', 'skill', 'kualifikasi']

processed_dfs_kaggle = []

for file_name, mapping in column_mapping.items():
    if file_name in dataframes_kaggle:
        df_temp = dataframes_kaggle[file_name]
        cols_to_extract = [col for col in mapping.keys() if col in df_temp.columns]
        df_extracted = df_temp[cols_to_extract].rename(columns=mapping)
        processed_dfs_kaggle.append(df_extracted)
        print(f"Berhasil memproses: {file_name} -> {len(df_extracted)} baris")
    else:
        print(f"[WARNING] {file_name} tidak ditemukan di memory.")

df_kaggle = pd.concat(processed_dfs_kaggle, ignore_index=True)

# Tambahkan kolom target yang belum ada
for col in target_columns_ai:
    if col not in df_kaggle.columns:
        df_kaggle[col] = pd.NA

df_kaggle = df_kaggle[target_columns_ai]

print(f"\n{'='*55}")
print("MERGE KAGGLE SELESAI!")
print(f"Total baris gabungan: {len(df_kaggle)}")
print(f"{'='*55}")
df_kaggle.head()

Berhasil memproses: IT_Job_Roles_Skills.xlsx -> 493 baris
Berhasil memproses: Job opportunities.xlsx -> 364 baris
Berhasil memproses: job_titles_and_descriptions.xlsx -> 289 baris
Berhasil memproses: jobs.xlsx -> 353 baris
Berhasil memproses: JobsDatasetProcessed.xlsx -> 3000 baris
Berhasil memproses: postings.xlsx -> 6025 baris
Berhasil memproses: Scrapped_data.xlsx -> 2250 baris

MERGE KAGGLE SELESAI!
Total baris gabungan: 12774


,posisi,pendidikan,pengalaman,gender,usia,skill,kualifikasi
0,.NET Developer,NaN,NaN,<NA>,<NA>,NaN,Develops applications using .NET framework tec...
1,2D ARTIST/ANIMATOR,NaN,NaN,<NA>,<NA>,NaN,Creates 2D animations for various media.
2,COMPUTER GRAPHICS ANIMATOR,NaN,NaN,<NA>,<NA>,NaN,Creates animations using computer graphics sof...
3,3D ARTIST/ANIMATOR,NaN,NaN,<NA>,<NA>,NaN,Creates 3D animations for various media.
4,TECHNICAL ACCOUNT MANAGER,NaN,NaN,<NA>,<NA>,NaN,Manages relationships with technical customers...


## 1.3 Cleaning Data Scraping (FD)

Dataset scraping dipersiapkan untuk kebutuhan **Full Data (FD)** yang mencakup semua kolom.
Proses cleaning meliputi: pengecekan nilai kosong, penghapusan baris invalid, pengisian nilai default,
penghapusan duplikat, dan normalisasi kolom kategorikal.

### 1.3.1. Cek Nilai Kosong

Menghitung jumlah dan persentase nilai kosong (NaN) per kolom.
Hanya kolom yang memiliki nilai kosong yang ditampilkan agar lebih mudah dibaca.

In [6]:
print("=== CEK NILAI KOSONG — SCRAPING (FD) ===")
missing = df_scraping.isnull().sum()
missing_pct = (df_scraping.isnull().sum() / len(df_scraping) * 100).round(2)
missing_df = pd.DataFrame({'jumlah_kosong': missing, 'persen_kosong (%)': missing_pct})
print(missing_df[missing_df['jumlah_kosong'] > 0].to_string())
print(f"\nTotal baris: {df_scraping.shape[0]}")

=== CEK NILAI KOSONG — SCRAPING (FD) ===
                 jumlah_kosong  persen_kosong (%)
tautan                     449               4.55
posisi                     449               4.55
perusahaan                 449               4.55
lokasi                     450               4.56
gaji                      2843              28.79
jenis_pekerjaan            449               4.55
kategori                   449               4.55
pendidikan                 577               5.84
pengalaman                 675               6.84
gender                    8285              83.90
usia                      7080              71.70
skill                      449               4.55
kualifikasi                449               4.55

Total baris: 9875


### 1.3.2. Drop Baris jika Kolom `tautan` Kosong

Baris yang tidak memiliki nilai pada kolom `tautan` dianggap tidak valid
karena `tautan` merupakan identitas unik dari setiap lowongan pekerjaan.
`dropna(subset=[...])` menghapus baris hanya berdasarkan kolom yang ditentukan.

In [7]:
before = df_scraping.shape[0]
df_scraping = df_scraping.dropna(subset=['tautan'])
after = df_scraping.shape[0]
print(f"Baris sebelum drop : {before}")
print(f"Baris setelah drop : {after}")
print(f"✅ Total baris dihapus: {before - after}")

Baris sebelum drop : 9875
Baris setelah drop : 9426
✅ Total baris dihapus: 449


### 1.3.3. Isi Nilai Kosong pada Kolom Tertentu

Kolom-kolom yang bersifat opsional diisi dengan nilai default yang deskriptif
agar tidak ada nilai kosong yang menyebabkan error pada proses selanjutnya.
`fillna()` menggantikan semua NaN dengan nilai yang ditentukan.

In [8]:
print("Nilai kosong SEBELUM fillna:")
print(df_scraping[['gaji', 'gender', 'usia', 'pengalaman', 'pendidikan']].isnull().sum())

df_scraping['gaji']       = df_scraping['gaji'].fillna('perusahaan tidak menampilkan gaji')
df_scraping['gender']     = df_scraping['gender'].fillna('tanpa ketentuan')
df_scraping['usia']       = df_scraping['usia'].fillna('tanpa batasan usia')
df_scraping['pengalaman'] = df_scraping['pengalaman'].fillna('tidak ada')
df_scraping['pendidikan'] = df_scraping['pendidikan'].fillna('terbuka untuk semua jenjang dan jurusan')

print("\nNilai kosong SETELAH fillna:")
print(df_scraping[['gaji', 'gender', 'usia', 'pengalaman', 'pendidikan']].isnull().sum())
print("\n✅ Nilai kosong telah diisi")

df_scraping.isnull().sum()

Nilai kosong SEBELUM fillna:
gaji          2394
gender        7836
usia          6631
pengalaman     226
pendidikan     128
dtype: int64

Nilai kosong SETELAH fillna:
gaji          0
gender        0
usia          0
pengalaman    0
pendidikan    0
dtype: int64

✅ Nilai kosong telah diisi


tautan             0
posisi             0
perusahaan         0
lokasi             1
gaji               0
jenis_pekerjaan    0
kategori           0
pendidikan         0
pengalaman         0
gender             0
usia               0
skill              0
kualifikasi        0
dtype: int64

### 1.3.4. Hapus Duplikat

Baris duplikat diidentifikasi berdasarkan kombinasi kolom `perusahaan` dan `posisi`,
karena satu perusahaan tidak seharusnya memiliki dua lowongan dengan posisi yang sama.
Setelah penghapusan, index di-reset agar berurutan kembali.

In [9]:
total_duplikat = df_scraping.duplicated(subset=['perusahaan', 'posisi']).sum()
print(f"Jumlah baris duplikat (perusahaan & posisi): {total_duplikat}")

before = df_scraping.shape[0]
df_scraping = df_scraping.drop_duplicates(subset=['perusahaan', 'posisi'])
after = df_scraping.shape[0]
print(f"Baris sebelum hapus duplikat : {before}")
print(f"Baris setelah hapus duplikat : {after}")
print(f"✅ Total duplikat dihapus     : {before - after}")
df_scraping.reset_index(drop=True, inplace=True)

Jumlah baris duplikat (perusahaan & posisi): 8861
Baris sebelum hapus duplikat : 9426
Baris setelah hapus duplikat : 565
✅ Total duplikat dihapus     : 8861


### 1.3.5. Normalisasi Kolom `jenis_pekerjaan` & `kategori`

Kolom `jenis_pekerjaan` dipecah menjadi dua kolom: `tipe_waktu_kerja` dan `sistem_kerja`
berdasarkan separator `·` atau `.` menggunakan `str.split()` dengan regex.

Kolom `kategori` diekstrak bagian keduanya (setelah tanda `>`) menjadi `kategori_peran`
untuk mendapatkan sub-kategori yang lebih spesifik.

In [10]:
# Pisah jenis_pekerjaan → tipe_waktu_kerja & sistem_kerja
df_scraping[['tipe_waktu_kerja', 'sistem_kerja']] = (
    df_scraping['jenis_pekerjaan']
    .str.split(r'\s*[·.]\s*', n=1, expand=True)
)
df_scraping = df_scraping.drop(columns=['jenis_pekerjaan'])
print("✅ Kolom jenis_pekerjaan dipecah menjadi tipe_waktu_kerja & sistem_kerja")

# Ekstrak kategori_peran dari kolom kategori
df_scraping['kategori_peran'] = df_scraping['kategori'].str.split(r'\s*>\s*', n=1, expand=True)[1]
df_scraping = df_scraping.drop(columns=['kategori'])
print("✅ Kolom kategori diekstrak menjadi kategori_peran")

print("\nSample hasil:")
df_scraping[['tipe_waktu_kerja', 'sistem_kerja', 'kategori_peran']].head()

✅ Kolom jenis_pekerjaan dipecah menjadi tipe_waktu_kerja & sistem_kerja
✅ Kolom kategori diekstrak menjadi kategori_peran

Sample hasil:


,tipe_waktu_kerja,sistem_kerja,kategori_peran
0,Penuh Waktu,Kerja di kantor,AI Engineer
1,Penuh Waktu,Remote/Dari rumah,Posisi Kecerdasan Buatan Lainnya
2,Kontrak,Kerja di kantor,AI Engineer
3,Penuh Waktu,Kerja di kantor,AI Engineer
4,Penuh Waktu,Kerja di kantor,AI Engineer


### 1.3.6. Cek Akhir Dataset Scraping (FD)

Menampilkan informasi final dataset scraping setelah seluruh proses cleaning selesai:
shape, tipe data, dan sisa nilai kosong yang belum tertangani.

In [11]:
print("=== INFO DATASET FINAL — SCRAPING (FD) ===")
df_scraping.info()
print(f"\nShape: {df_scraping.shape}")
print("\nNilai kosong tersisa:")
print(df_scraping.isnull().sum()[df_scraping.isnull().sum() > 0])
print("\nSample data:")
df_scraping.head()

=== INFO DATASET FINAL — SCRAPING (FD) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 565 entries, 0 to 564
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   tautan            565 non-null    object
 1   posisi            565 non-null    object
 2   perusahaan        565 non-null    object
 3   lokasi            565 non-null    object
 4   gaji              565 non-null    object
 5   pendidikan        565 non-null    object
 6   pengalaman        565 non-null    object
 7   gender            565 non-null    object
 8   usia              565 non-null    object
 9   skill             565 non-null    object
 10  kualifikasi       565 non-null    object
 11  tipe_waktu_kerja  565 non-null    object
 12  sistem_kerja      565 non-null    object
 13  kategori_peran    565 non-null    object
dtypes: object(14)
memory usage: 61.9+ KB

Shape: (565, 14)

Nilai kosong tersisa:
Series([], dtype: int64)

Sample 

,tautan,posisi,perusahaan,lokasi,gaji,pendidikan,pengalaman,gender,usia,skill,kualifikasi,tipe_waktu_kerja,sistem_kerja,kategori_peran
0,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Minimal Sarjana (S1),5 - 10 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr...",Penuh Waktu,Kerja di kantor,AI Engineer
1,https://glints.com/id/opportunities/jobs/ai-cr...,AI Creative Engineer,Willscale,"Jakarta Selatan, DKI Jakarta",Rp4.500.000 - 6.000.000/Bulan,Minimal Sarjana (S1),1 - 3 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"Creative Thinking, Artificial Intelligence, Vi...",Are you a tech-driven innovator who loves to p...,Penuh Waktu,Remote/Dari rumah,Posisi Kecerdasan Buatan Lainnya
2,https://glints.com/id/opportunities/jobs/ai-en...,AI Engineer,PT DIGITAL SEJAHTERA NUSANTARA,"Jakarta Pusat, DKI Jakarta",Rp7.000.000 - 11.000.000/Bulan,Minimal Sarjana (S1),1 - 3 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"Pattern Recognition, Machine Learning, Artific...",Requirements:\nHands-on experience implementin...,Kontrak,Kerja di kantor,AI Engineer
3,https://glints.com/id/opportunities/jobs/ai-en...,AI Engineer,PT Kotagede Jewellery Group,"Kab. Sleman, DI Yogyakarta",Rp4.000.000 - 5.500.000/Bulan,Minimal Sarjana (S1),1 - 3 tahun pengalaman,tanpa ketentuan,25-35 tahun,"Artificial Intelligence, Python, Data Mining, ...",Kami adalah perusahaan dengan tim yang diisi o...,Penuh Waktu,Kerja di kantor,AI Engineer
4,https://glints.com/id/opportunities/jobs/ai-en...,AI Engineer,Zando Agency,"Jakarta Utara, DKI Jakarta",Rp13.000.000 - 17.000.000/Bulan,Minimal Sarjana (S1),3 - 5 tahun pengalaman,tanpa ketentuan,20-30 tahun,"Machine Learning, Cloud Platform System, Pytho...",Kamu siap bangun masa depan bareng tim yang ge...,Penuh Waktu,Kerja di kantor,AI Engineer


## 1.4 Cleaning Dataset Kaggle (AI)

### 1.4.1. Gabungkan Data Scraping (Clean) dengan Data Kaggle

Sebelum masuk proses cleaning AI, kolom yang relevan dari `df_scraping` yang sudah bersih
digabungkan ke `df_kaggle` agar data scraping juga ikut digunakan untuk training model AI.
Kolom yang tidak tersedia di `df_scraping` akan diisi `pd.NA` agar struktur tetap seragam.

In [12]:
# Kolom AI yang akan diambil dari df_scraping
KOLOM_AI = ['posisi', 'pendidikan', 'pengalaman', 'gender', 'usia', 'skill', 'kualifikasi']

# Ambil kolom yang ada di df_scraping
kolom_tersedia = [col for col in KOLOM_AI if col in df_scraping.columns]
df_scraping_ai = df_scraping[kolom_tersedia].copy()

# Tambahkan kolom yang belum ada
for col in KOLOM_AI:
    if col not in df_scraping_ai.columns:
        df_scraping_ai[col] = pd.NA

df_scraping_ai = df_scraping_ai[KOLOM_AI]

# Gabungkan ke df_kaggle
df_kaggle = pd.concat([df_kaggle, df_scraping_ai], ignore_index=True)

print(f"✅ Data scraping berhasil digabungkan ke df_kaggle")
print(f"   Baris dari scraping  : {len(df_scraping_ai)}")
print(f"   Total df_kaggle kini : {len(df_kaggle)} baris, {df_kaggle.shape[1]} kolom")
display(df_kaggle.tail(5))

✅ Data scraping berhasil digabungkan ke df_kaggle
   Baris dari scraping  : 565
   Total df_kaggle kini : 13339 baris, 7 kolom


,posisi,pendidikan,pengalaman,gender,usia,skill,kualifikasi
13334,Web Developer,Minimal SMA/SMK,1 - 3 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"jQuery, MySQL, JavaScript, PHP, CodeIgniter, B...",💼 WE ARE HIRING: Web Developer\nPerusahaan: Br...
13335,Web Developer,Minimal SMA/SMK,Pengalaman kurang dari 1 tahun,tanpa ketentuan,18-25 tahun,"MySQL, jQuery, JavaScript",🚀 Lowongan Freelance – Fullstack Web Developer...
13336,Web Developer ( freelance),Minimal Sarjana (S1),1 - 3 tahun pengalaman,Laki-laki saja,tanpa batasan usia,"Python, Microsoft SQL Server, PostgreSQL",sudah bisa menguasai
13337,Web Developer Intern,Minimal Diploma (D1 - D4),Pengalaman kurang dari 1 tahun,tanpa ketentuan,tanpa batasan usia,"Time Management, Python, Java, Teamwork, Techn...",Note : Nominal uang saku yang tercantum merupa...
13338,Web Developer Intern,Minimal SMA/SMK,Pengalaman kurang dari 1 tahun,tanpa ketentuan,tanpa batasan usia,"PHP, Laravel, MySQL",Responsibilities :\n-) Assist in developing an...


### 1.4.2. Cek Nilai Kosong

Menampilkan jumlah nilai kosong per kolom sebelum proses cleaning data Kaggle dimulai.

In [13]:
print(">>> CEK KOLOM KOSONG SEBELUM CLEANING — KAGGLE (AI):")
display(df_kaggle.isnull().sum())
print("\n" + "="*55)

>>> CEK KOLOM KOSONG SEBELUM CLEANING — KAGGLE (AI):


posisi             1
pendidikan     11211
pengalaman      2578
gender         12774
usia           12774
skill           1568
kualifikasi      718
dtype: int64

### 1.4.3. Drop & Fill Nilai Kosong

Baris yang tidak memiliki nilai pada kolom `posisi` atau `skill` dihapus karena keduanya
merupakan kolom inti yang wajib ada untuk keperluan training AI.
Kolom lain yang kosong diisi dengan nilai default yang deskriptif.

In [14]:
df_ai = df_kaggle.copy()

# Drop baris jika posisi atau skill kosong
df_ai.dropna(subset=['posisi', 'skill'], inplace=True)

# Isi kolom kosong dengan nilai default
fill_values_ai = {
    'gender'     : 'tanpa ketentuan',
    'usia'       : 'tanpa batasan usia',
    'pengalaman' : 'tidak ada',
    'pendidikan' : 'terbuka untuk semua jenjang dan jurusan'
}
df_ai.fillna(value=fill_values_ai, inplace=True)

print("✅ Drop & fillna selesai")
print(f"Baris setelah drop posisi/skill kosong: {len(df_ai)}")

✅ Drop & fillna selesai
Baris setelah drop posisi/skill kosong: 11770


### 1.4.4. Hapus Duplikat

Baris duplikat dihapus berdasarkan kombinasi `posisi` dan `skill`.
Duplikat yang dipertahankan adalah yang pertama muncul (`keep='first'`).
Index di-reset setelah penghapusan.

In [15]:
before = df_ai.shape[0]
df_ai.drop_duplicates(subset=['posisi', 'skill'], keep='first', inplace=True)
df_ai.reset_index(drop=True, inplace=True)
after = df_ai.shape[0]
print(f"Baris sebelum hapus duplikat : {before}")
print(f"Baris setelah hapus duplikat : {after}")
print(f"✅ Total duplikat dihapus     : {before - after}")

Baris sebelum hapus duplikat : 11770
Baris setelah hapus duplikat : 11027
✅ Total duplikat dihapus     : 743


## 1.5 Standardisasi Kolom `pendidikan`

Fungsi `standardize_pendidikan()` memetakan berbagai variasi teks jenjang pendidikan
(dari Bahasa Indonesia maupun Inggris) ke dalam format standar yang seragam.
Pencocokan dilakukan secara case-insensitive menggunakan `str.lower()`.
Jika tidak ada jenjang yang cocok, dikembalikan nilai default.

In [16]:
# Standarisasi kolom pendidikan berdasarkan tingkat jenjang. 
# Jika tidak ditemukan jenjang yang relevan, diisi nilai default.
def standardize_pendidikan(text):
  
    DEFAULT_VALUE = 'terbuka untuk semua jenjang dan jurusan'
    
    if pd.isna(text):
        return DEFAULT_VALUE
        
    text_lower = str(text).strip().lower()
    
    # Cek nilai default eksplisit
    if text_lower in ['terbuka untuk semua jentang dan jurusan', 
                      'terbuka untuk semua jenjang dan jurusan']:
        return DEFAULT_VALUE

    # Mapping keywords ke jenjang pendidikan (urut dari tertinggi ke terendah)
    education_map = {
        'Doktoral (S3)': ['phd', 'ph.d', 'doktor', 'doctoral', 's3', 's-3'],
        'Magister (S2)': [
            'master', 'magister', 's2', 's-2', 'postgraduate', 
            'post-graduate', 'graduate degree', 'mba', 'mpa', 'm.s', 'ms in'
        ],
        'Sarjana (S1)': [
            'bachelor', 'sarjana', 's1', 's-1', 'undergraduate', 
            "bachelor's", 'bs degree', 'b.s', 'ba degree', 'b.a', 
            'college degree', '4-year degree', 'degree in', 'university degree'
        ],
        'Diploma IV (D4)': ['d4', 'd-4', 'diploma iv', 'diploma 4'],
        'Diploma III (D3)': [
            'd3', 'd-3', 'diploma iii', 'diploma 3', 
            'associate degree', "associate's"
        ],
        'Diploma II (D2)': ['d2', 'd-2', 'diploma ii', 'diploma 2'],
        'Diploma I (D1)': ['d1', 'd-1', 'diploma i', 'diploma 1'],
        'SMA/SMK': [
            'sma', 'smk', 'high school', 'secondary school', 
            'ged', 'vocational school', 'high school diploma', 'diploma'
        ],
        'SMP/SD': ['smp', 'middle school', 'junior high', 'sd']
    }

    # Iterasi pencarian kata kunci
    for level, keywords in education_map.items():
        if any(k in text_lower for k in keywords):
            return level

    return DEFAULT_VALUE


# --- Eksekusi pada DataFrame ---
# Pastikan df_ai sudah didefinisikan sebelumnya
df_ai['pendidikan'] = df_ai['pendidikan'].apply(standardize_pendidikan)

# --- Cetak Hasil ---
print(">>> STANDARDISASI PENDIDIKAN SELESAI!")
print("=" * 55)
print("Distribusi Jenjang Pendidikan:")
print(df_ai['pendidikan'].value_counts())
print("=" * 55, "\n")

>>> STANDARDISASI PENDIDIKAN SELESAI!
Distribusi Jenjang Pendidikan:
pendidikan
terbuka untuk semua jenjang dan jurusan    9459
Sarjana (S1)                                961
SMA/SMK                                     198
Magister (S2)                               191
Diploma IV (D4)                             126
Doktoral (S3)                                80
Diploma III (D3)                             10
SMP/SD                                        1
Diploma I (D1)                                1
Name: count, dtype: int64



## 1.6 Standardisasi Kolom `pengalaman`

Fungsi `standardize_pengalaman()` mengkonversi berbagai format teks pengalaman kerja
ke format standar `X-Y tahun` atau `X+ tahun`.

Urutan pencocokan:
1. **Level map** — kata kunci level (Junior, Senior, Entry-Level) → rentang tahun
2. **Range pattern** — pola `X-Y years` → `X-Y tahun`
3. **Plus pattern** — pola `X+ years` → `X+ tahun`
4. **Single pattern** — pola `X years` → `X tahun`
5. Jika tidak ada pola yang cocok → `tidak ada`

In [17]:
# Standarisasi kolom pengalaman menjadi format 'X-Y tahun' atau 'X+ tahun'. 
# Nilai level (Entry-Level, Junior, dll) dikonversi ke rentang tahun.
# Jika tidak ditemukan angka tahun, diisi 'tidak ada'.
  
def standardize_pengalaman(text):
  
    if pd.isna(text):
        return 'tidak ada'

    text_str = str(text).strip()
    text_lower = text_str.lower()

    level_map = {
        'entry-level': '0-1 tahun',
        'entry level': '0-1 tahun',
        'junior'     : '1-3 tahun',
        'mid-level'  : '3-5 tahun',
        'mid level'  : '3-5 tahun',
        'senior'     : '5+ tahun',
        'internship' : '0 tahun (magang)',
        'intern'     : '0 tahun (magang)',
        'tidak ada'  : 'tidak ada',
    }
    if text_lower in level_map:
        return level_map[text_lower]

    range_pat = re.search(r'(\d+)\s*[-–]\s*(\d+)\s*(years?|yrs?|tahun)', text_lower)
    if range_pat:
        return f'{range_pat.group(1)}-{range_pat.group(2)} tahun'

    plus_pat = re.search(r'(\d+)\s*\+\s*(years?|yrs?|tahun)', text_lower)
    if plus_pat:
        return f'{plus_pat.group(1)}+ tahun'

    plus_before = re.search(r'\+\s*(\d+)\s*(years?|yrs?|tahun)', text_lower)
    if plus_before:
        return f'{plus_before.group(1)}+ tahun'

    single_pat = re.search(r'(\d+)\s*(years?|yrs?|tahun)', text_lower)
    if single_pat:
        return f'{single_pat.group(1)} tahun'

    return 'tidak ada'


df_ai['pengalaman'] = df_ai['pengalaman'].apply(standardize_pengalaman)

print(">>> STANDARDISASI PENGALAMAN SELESAI!")
print(f"{'='*55}")
print("Distribusi Pengalaman (Top 20):")
print(df_ai['pengalaman'].value_counts().head(20))
print(f"{'='*55}\n")

>>> STANDARDISASI PENGALAMAN SELESAI!
Distribusi Pengalaman (Top 20):
pengalaman
tidak ada     8277
1-3 tahun      392
0-3 tahun      348
3-5 tahun      268
2-5 tahun      179
3-6 tahun      126
5-8 tahun      122
1 tahun         92
4-7 tahun       87
1-4 tahun       75
5-10 tahun      69
2-3 tahun       64
0-1 tahun       63
3 tahun         55
5 tahun         51
2-4 tahun       44
2 tahun         35
6-9 tahun       35
8-11 tahun      31
1-5 tahun       29
Name: count, dtype: int64



## 1.7 Cek Akhir & Preview Data AI

Menampilkan kondisi final `df_ai` setelah seluruh proses cleaning Bagian 1 selesai:
jumlah nilai kosong, struktur kolom, dan preview 10 baris pertama.

In [18]:
print(">>> CEK KOLOM KOSONG SETELAH PREPROCESSING:")
display(df_ai.isnull().sum())
print("\n" + "="*55)

print(">>> INFO DATASET AI:")
df_ai.info()
print("\n" + "="*55)

print(">>> PREVIEW DATA AI (HEAD):")
display(df_ai.head(10))

>>> CEK KOLOM KOSONG SETELAH PREPROCESSING:


posisi           0
pendidikan       0
pengalaman       0
gender           0
usia             0
skill            0
kualifikasi    352
dtype: int64


>>> INFO DATASET AI:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11027 entries, 0 to 11026
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   posisi       11027 non-null  object
 1   pendidikan   11027 non-null  object
 2   pengalaman   11027 non-null  object
 3   gender       11027 non-null  object
 4   usia         11027 non-null  object
 5   skill        11027 non-null  object
 6   kualifikasi  10675 non-null  object
dtypes: object(7)
memory usage: 603.2+ KB

>>> PREVIEW DATA AI (HEAD):


,posisi,pendidikan,pengalaman,gender,usia,skill,kualifikasi
0,Software Engineer,terbuka untuk semua jenjang dan jurusan,0-1 tahun,tanpa ketentuan,tanpa batasan usia,"Java, Python",Develop software
1,Data Analyst,terbuka untuk semua jenjang dan jurusan,1-3 tahun,tanpa ketentuan,tanpa batasan usia,"SQL, Excel",Analyze data
2,Network Engineer,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"Cisco, WAN",Maintain networks
3,Cloud Architect,terbuka untuk semua jenjang dan jurusan,5+ tahun,tanpa ketentuan,tanpa batasan usia,"AWS, Azure",Design cloud
4,Cybersecurity Analyst,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,Cybersecurity,Protect data
5,IT Project Manager,terbuka untuk semua jenjang dan jurusan,5+ tahun,tanpa ketentuan,tanpa batasan usia,"PMP, Agile",Manage projects
6,Data Scientist,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"R, Python",Analyze big data
7,DevOps Engineer,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"Docker, Jenkins",Streamline devops
8,IT Support Analyst,terbuka untuk semua jenjang dan jurusan,0-1 tahun,tanpa ketentuan,tanpa batasan usia,"Windows, Linux",Provide IT support
9,UX/UI Designer,terbuka untuk semua jenjang dan jurusan,1-3 tahun,tanpa ketentuan,tanpa batasan usia,UI/UX Design,Design user interfaces


---
# BAGIAN 2 — Cleaning Kolom `kualifikasi` via Groq API

Bagian ini membersihkan dan memperkaya kolom `kualifikasi` yang berisi teks singkat
seperti *"Develop software"* menjadi deskripsi kualifikasi yang **detail, spesifik, dan kontekstual**
menggunakan **Groq API** dengan model `llama-3.1-8b-instant`.

Proses dijalankan dengan sistem checkpoint setiap 50 baris agar dapat dilanjutkan
jika notebook restart atau terjadi gangguan.

## 2.1 Preview Kolom `kualifikasi` (Sebelum Cleaning)

Menampilkan kondisi awal kolom `kualifikasi` — biasanya berisi teks yang sangat singkat
dan belum cukup informatif sebagai kualifikasi pekerjaan.

In [19]:
pd.set_option('display.max_colwidth', 200)

print(f"Total baris        : {len(df_ai):,}")
print(f"Nilai kosong       : {df_ai['kualifikasi'].isna().sum()}")
print(f"Rata-rata panjang  : {df_ai['kualifikasi'].dropna().apply(len).mean():.1f} karakter")
print()

df_ai[['posisi', 'skill', 'kualifikasi']].head(10)

Total baris        : 11,027
Nilai kosong       : 352
Rata-rata panjang  : 3117.8 karakter



,posisi,skill,kualifikasi
0,Software Engineer,"Java, Python",Develop software
1,Data Analyst,"SQL, Excel",Analyze data
2,Network Engineer,"Cisco, WAN",Maintain networks
3,Cloud Architect,"AWS, Azure",Design cloud
4,Cybersecurity Analyst,Cybersecurity,Protect data
5,IT Project Manager,"PMP, Agile",Manage projects
6,Data Scientist,"R, Python",Analyze big data
7,DevOps Engineer,"Docker, Jenkins",Streamline devops
8,IT Support Analyst,"Windows, Linux",Provide IT support
9,UX/UI Designer,UI/UX Design,Design user interfaces


## 2.2 Input API Key & Konfigurasi Groq

Masukkan **Groq API Key** menggunakan `getpass` agar tidak tampil di layar.
Dapatkan API key gratis di: https://console.groq.com

Konfigurasi:
- `DELAY_BETWEEN_CALLS` — jeda antar panggilan API untuk menghindari rate limit
- `MAX_INPUT_CHARS` — batas karakter input per prompt agar tidak melebihi context window
- `MAX_TOKENS` — batas panjang output yang dihasilkan model

In [20]:
GROQ_API_KEY = getpass.getpass("Masukkan GROQ API KEY: ")
print("API key berhasil diinput ✓")

API key berhasil diinput ✓


In [21]:
# ── Konfigurasi ──────────────────────────────────────────────────────────────
DELAY_BETWEEN_CALLS = 1.5    # detik antar panggilan API (hindari rate limit)
MAX_INPUT_CHARS     = 800    # batas karakter input per prompt
MAX_TOKENS          = 250    # batas token output per panggilan

# Inisialisasi Groq client
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq client siap digunakan ✓")
print(f"Model    : llama-3.1-8b-instant")
print(f"Delay    : {DELAY_BETWEEN_CALLS}s per panggilan")
print(f"Max char : {MAX_INPUT_CHARS} karakter input")

Groq client siap digunakan ✓
Model    : llama-3.1-8b-instant
Delay    : 1.5s per panggilan
Max char : 800 karakter input


## 2.3 Fungsi `build_prompt()`

Membangun prompt untuk Groq dari satu baris data lowongan kerja.
Konteks posisi, skill, pengalaman, pendidikan, dan kualifikasi asli digabungkan
agar model menghasilkan output yang relevan dan spesifik untuk posisi tersebut.

Mengembalikan `None` jika data tidak cukup (posisi kosong atau teks terlalu pendek).

In [22]:
# Buat prompt untuk Groq dari satu baris data lowongan kerja.
def build_prompt(row: pd.Series) -> str | None:

    posisi      = str(row.get('posisi', '')).strip()
    kualifikasi = str(row.get('kualifikasi', '')).strip()
    skill       = str(row.get('skill', '')).strip()
    pengalaman  = str(row.get('pengalaman', '')).strip()
    pendidikan  = str(row.get('pendidikan', '')).strip()

    # Lewati baris yang tidak informatif
    if not posisi or posisi.lower() == 'nan':
        return None
    if len(kualifikasi) < 3 and len(skill) < 3:
        return None

    # Susun konteks — batasi panjang tiap bagian
    konteks = (
        f"Posisi      : {posisi[:100]}\n"
        f"Kualifikasi : {kualifikasi[:MAX_INPUT_CHARS]}\n"
        f"Skill       : {skill[:200]}\n"
        f"Pengalaman  : {pengalaman[:100]}\n"
        f"Pendidikan  : {pendidikan[:150]}"
    )

    return f"""Kamu adalah spesialis rekrutmen IT. Di bawah ini adalah data mentah lowongan pekerjaan dari sebuah sistem HR.

DATA LOWONGAN:
{konteks}

TUGAS:
Ekstrak dan tulis ulang kualifikasi dalam SATU kalimat panjang yang mengalir.
Struktur wajib: [skill teknis spesifik] + [lama & bidang pengalaman] + [tugas utama pekerjaan].
Gunakan kata kerja aktif: menguasai, mampu, berpengalaman, bertanggung jawab.
Bahasa Indonesia. Tanpa bullet. Tanpa enter. Tanpa kalimat pembuka/penutup.
Jika data tidak cukup → tulis: NULL
OUTPUT:
"""

print("Fungsi build_prompt() siap ✓")

Fungsi build_prompt() siap ✓


## 2.4 Fungsi `extract_kualifikasi_groq()` & `reinput_api_key()`

Fungsi `extract_kualifikasi_groq()` memanggil Groq API dan mengembalikan hasil cleaning.
Temperature rendah (0.3) digunakan agar output konsisten dan tidak terlalu kreatif.

Fungsi `reinput_api_key()` dipanggil secara otomatis saat terjadi error 429 (rate limit),
meminta user untuk memasukkan API key baru tanpa menghentikan proses.

In [23]:
# Minta user memasukkan Groq API key baru saat kena rate limit 429.
def reinput_api_key() -> None:

    global groq_client
    print("\nRate limit tercapai! API key saat ini sudah habis kuotanya.")
    new_key = getpass.getpass("Masukkan GROQ API KEY baru: ")
    groq_client = Groq(api_key=new_key)
    print("✓ API key baru berhasil diset. Melanjutkan proses...\n")

# Panggil Groq API. Jika kena 429, minta API key baru lalu retry.
def extract_kualifikasi_groq(prompt: str, retry: int = 3) -> str:

    if retry == 0:
        print("  ✗ Gagal setelah beberapa percobaan, baris dikosongkan.")
        return ''

    try:
        response = groq_client.chat.completions.create(
            messages=[{'role': 'user', 'content': prompt}],
            model='llama-3.1-8b-instant',
            temperature=0.3,
            max_tokens=MAX_TOKENS,
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        err = str(e)
        if '429' in err or 'rate_limit' in err.lower() or 'rate limit' in err.lower():
            reinput_api_key()
            time.sleep(2)
            return extract_kualifikasi_groq(prompt, retry=retry - 1)
        else:
            print(f"  ✗ Error: {err[:100]}")
            return ''

print("Fungsi extract_kualifikasi_groq() + reinput_api_key() siap ✓")

Fungsi extract_kualifikasi_groq() + reinput_api_key() siap ✓


## 2.5 Fungsi `clean_kualifikasi()` — Wrapper Utama

Wrapper yang menggabungkan `build_prompt()` dan `extract_kualifikasi_groq()`.
Jika data tidak valid (prompt = None), dikembalikan string kosong tanpa memanggil API
sehingga tidak membuang kuota API.

In [24]:
# Wrapper: bangun prompt → panggil Groq → kembalikan hasil.
def clean_kualifikasi(row: pd.Series) -> str:

    prompt = build_prompt(row)
    if prompt is None:
        return ''
    return extract_kualifikasi_groq(prompt)

print("Fungsi clean_kualifikasi() siap ✓")

Fungsi clean_kualifikasi() siap ✓


## 2.6 Uji Coba pada 3 Baris Pertama

Jalankan cell ini **sebelum** memproses seluruh data untuk memastikan
prompt dan koneksi API berjalan dengan benar.
Menampilkan perbandingan kualifikasi asli vs hasil cleaning dari Groq.

In [25]:
print("=" * 60)
print("  UJI COBA 3 BARIS PERTAMA")
print("=" * 60)

for index, row in df_ai.head(3).iterrows():
    kual_asli = str(row.get('kualifikasi', '')).strip()
    posisi    = str(row.get('posisi', '')).strip()

    print(f"\n[Baris {index}] Posisi      : {posisi}")
    print(f"[Baris {index}] Kual. asli  : {kual_asli}")

    hasil = clean_kualifikasi(row)
    print(f"[Baris {index}] Kual. bersih: {hasil}")
    print("-" * 60)

    time.sleep(DELAY_BETWEEN_CALLS)

print("\n✅ Uji coba selesai — siap proses seluruh data!")

  UJI COBA 3 BARIS PERTAMA

[Baris 0] Posisi      : Software Engineer
[Baris 0] Kual. asli  : Develop software
[Baris 0] Kual. bersih: Menguasai Java dan Python, berpengalaman dalam bidang pengembangan perangkat lunak selama 0-1 tahun, bertanggung jawab untuk mengembangkan perangkat lunak yang efektif dan efisien.
------------------------------------------------------------

[Baris 1] Posisi      : Data Analyst
[Baris 1] Kual. asli  : Analyze data
[Baris 1] Kual. bersih: Menguasai SQL dan Excel, berpengalaman 1-3 tahun dalam bidang analisis data, bertanggung jawab untuk menganalisis data dan menyajikannya dalam bentuk yang efektif.
------------------------------------------------------------

[Baris 2] Posisi      : Network Engineer
[Baris 2] Kual. asli  : Maintain networks
[Baris 2] Kual. bersih: Menguasai teknologi jaringan Cisco dan WAN, berpengalaman dalam bidang jaringan selama 3-5 tahun, bertanggung jawab untuk menjaga kestabilan dan kinerja jaringan.
----------------------------

## 2.7 Proses Seluruh Data dengan Checkpoint

Proses seluruh baris secara aman dengan fitur:
- **Checkpoint CSV** — disimpan setiap `BATCH_SIZE` baris. Jika notebook restart, proses dilanjutkan dari titik terakhir secara otomatis.
- **Rate limit 429** — jika API key habis kuota, otomatis meminta key baru tanpa menghentikan proses.
- **Progress log** — ditampilkan setiap 10 baris agar proses dapat dipantau.

Kolom `kualifikasi_asli` disimpan sebagai backup sebelum kolom `kualifikasi` ditimpa.

In [26]:
# Konfigurasi checkpoint dan batch processing
CHECKPOINT_FILE = 'kualifikasi_checkpoint.csv'
BATCH_SIZE = 50

# Backup kolom asli (sekali saja)
if 'kualifikasi_asli' not in df_ai.columns:
    df_ai['kualifikasi_asli'] = df_ai['kualifikasi'].copy()

# Load checkpoint jika ada, dan buat set indeks yang sudah diproses
if os.path.exists(CHECKPOINT_FILE):
    ckpt_df = pd.read_csv(CHECKPOINT_FILE)
    ckpt_df = ckpt_df.drop_duplicates(subset='original_index', keep='last')
    done_indices = set(ckpt_df['original_index'].astype(int).tolist())
    print(f"📂 Checkpoint ditemukan")
    print(f"   Total selesai : {len(done_indices):,} baris")
    print(f"   Melanjutkan proses...\n")
else:
    ckpt_df = pd.DataFrame(columns=['original_index', 'kualifikasi'])
    done_indices = set()
    print("Belum ada checkpoint, mulai dari awal.\n")

# Hitung progress awal  
total   = len(df_ai)
pending = [idx for idx in df_ai.index if idx not in done_indices]
new_rows = []
success_count = len(done_indices)
failed_count  = 0
empty_count   = 0

print("=" * 60)
print(f"Total data      : {total:,}")
print(f"Sudah diproses  : {len(done_indices):,}")
print(f"Sisa data       : {len(pending):,}")
print(f"Batch size      : {BATCH_SIZE}")
print("=" * 60)

# Loop processing dengan progress dan checkpoint
for count, idx in enumerate(pending, start=1):
    row    = df_ai.loc[idx]
    posisi = str(row.get('posisi', '')).strip()

    processed = len(done_indices) + len(new_rows)

    if count % 10 == 0 or count == 1:
        pct = (processed / total) * 100
        print(f"[{processed:>5}/{total}] ({pct:.1f}%) {posisi[:50]}")

    try:
        hasil = clean_kualifikasi(row)
        if hasil is None:
            hasil = ''
        hasil = str(hasil).strip()
        if hasil == '':
            empty_count += 1
            print(f"Kosong | index={idx}")
        new_rows.append({'original_index': idx, 'kualifikasi': hasil})

    except Exception as e:
        failed_count += 1
        print(f"Error index={idx} | posisi: {posisi} | {str(e)}")
        continue

    # Save checkpoint setiap BATCH_SIZE baris
    if len(new_rows) >= BATCH_SIZE:
        batch_df = pd.DataFrame(new_rows)
        ckpt_df  = pd.concat([ckpt_df, batch_df], ignore_index=True)
        ckpt_df  = ckpt_df.drop_duplicates(subset='original_index', keep='last')
        ckpt_df.to_csv(CHECKPOINT_FILE, index=False)
        done_indices.update(batch_df['original_index'].tolist())
        success_count = len(done_indices)
        print(f"Checkpoint disimpan | Total saved: {success_count:,}")
        new_rows = []

    time.sleep(DELAY_BETWEEN_CALLS)

# Save sisa terakhir
if new_rows:
    batch_df = pd.DataFrame(new_rows)
    ckpt_df  = pd.concat([ckpt_df, batch_df], ignore_index=True)
    ckpt_df  = ckpt_df.drop_duplicates(subset='original_index', keep='last')
    ckpt_df.to_csv(CHECKPOINT_FILE, index=False)
    done_indices.update(batch_df['original_index'].tolist())
    print(f"Checkpoint terakhir disimpan | Total saved: {len(done_indices):,}")

# Gabungkan hasil ke dataframe utama
ckpt_indexed = (
    ckpt_df
    .drop_duplicates(subset='original_index', keep='last')
    .set_index('original_index')['kualifikasi']
)
df_ai['kualifikasi'] = df_ai.index.to_series().map(ckpt_indexed).fillna('')

# Statistik akhir
filled_count = df_ai['kualifikasi'].astype(str).str.strip().ne('').sum()
empty_final  = total - filled_count

print("\n" + "=" * 60)
print("✅ PROCESS SELESAI")
print("=" * 60)
print(f"Total data        : {total:,}")
print(f"Berhasil diproses : {len(done_indices):,}")
print(f"Gagal proses      : {failed_count:,}")
print(f"Hasil kosong      : {empty_count:,}")
print(f"Kolom terisi      : {filled_count:,}")
print(f"Kolom kosong      : {empty_final:,}")

📂 Checkpoint ditemukan
   Total selesai : 11,027 baris
   Melanjutkan proses...

Total data      : 11,027
Sudah diproses  : 11,027
Sisa data       : 0
Batch size      : 50

✅ PROCESS SELESAI
Total data        : 11,027
Berhasil diproses : 11,027
Gagal proses      : 0
Hasil kosong      : 0
Kolom terisi      : 11,024
Kolom kosong      : 3


## 2.8 Preview Hasil Cleaning Kualifikasi

Membandingkan kolom `kualifikasi_asli` (sebelum) dengan `kualifikasi` (sesudah Groq)
serta menampilkan statistik rata-rata panjang karakter untuk melihat seberapa besar
peningkatan kualitas teks yang dihasilkan.

In [27]:
pd.set_option('display.max_colwidth', 300)

print(f"Total baris diproses   : {len(df_ai):,}")
print(f"Rata-rata panjang baru : {df_ai['kualifikasi'].str.len().mean():.1f} karakter")
print(f"Rata-rata panjang lama : {df_ai['kualifikasi_asli'].str.len().mean():.1f} karakter")
print()

df_ai[['posisi', 'skill', 'kualifikasi_asli', 'kualifikasi']].head(10)

Total baris diproses   : 11,027
Rata-rata panjang baru : 437.5 karakter
Rata-rata panjang lama : 3117.8 karakter



,posisi,skill,kualifikasi_asli,kualifikasi
0,Software Engineer,"Java, Python",Develop software,"Menguasai Java dan Python, mampu mengembangkan software dengan pengalaman 0-1 tahun di bidang pengembangan perangkat lunak, bertanggung jawab untuk mengembangkan software."
1,Data Analyst,"SQL, Excel",Analyze data,"Menguasai SQL dan Excel, berpengalaman 1-3 tahun dalam bidang analisis data, bertanggung jawab untuk menganalisis data dan menyajikannya dalam bentuk yang efektif."
2,Network Engineer,"Cisco, WAN",Maintain networks,"Menguasai teknologi Cisco dan WAN, berpengalaman dalam bidang jaringan selama 3-5 tahun, bertanggung jawab untuk menjaga dan memelihara jaringan, serta mampu melakukan perawatan dan perbaikan jaringan."
3,Cloud Architect,"AWS, Azure",Design cloud,"Menguasai teknologi cloud seperti AWS dan Azure, berpengalaman lebih dari 5 tahun dalam bidang desain cloud, bertanggung jawab untuk merancang dan mengimplementasikan solusi cloud yang efektif dan efisien."
4,Cybersecurity Analyst,Cybersecurity,Protect data,"Menguasai Cybersecurity, berpengalaman 3-5 tahun di bidang keamanan siber, dan bertanggung jawab untuk melindungi data."
5,IT Project Manager,"PMP, Agile",Manage projects,"Menguasai PMP dan Agile, berpengalaman lebih dari 5 tahun dalam bidang IT, bertanggung jawab untuk mengelola proyek dan mampu mengatur tim untuk mencapai tujuan proyek."
6,Data Scientist,"R, Python",Analyze big data,"Menguasai R dan Python, berpengalaman 3-5 tahun dalam bidang analisis data, bertanggung jawab untuk menganalisis big data dan mengembangkan solusi yang efektif."
7,DevOps Engineer,"Docker, Jenkins",Streamline devops,"Menguasai Docker dan Jenkins, berpengalaman 3-5 tahun dalam bidang DevOps, bertanggung jawab untuk mengintegrasikan dan memantau proses pengembangan dan pengelolaan infrastruktur."
8,IT Support Analyst,"Windows, Linux",Provide IT support,"Menguasai Windows dan Linux, berpengalaman dalam bidang IT dengan pengalaman kerja 0-1 tahun, bertanggung jawab untuk memberikan dukungan teknis yang efektif sebagai IT Support Analyst."
9,UX/UI Designer,UI/UX Design,Design user interfaces,"UX/UI Designer yang berpengalaman menguasai UI/UX Design, mampu mengembangkan desain user interfaces yang efektif dan efisien, serta bertanggung jawab dalam mengelola proyek desain dengan lama pengalaman 1-3 tahun di bidang desain."


---
# BAGIAN 3 — Cleaning & Standarisasi Kolom `skill`

Bagian ini membersihkan dan menstandarisasi nilai pada kolom `skill` menggunakan kamus standarisasi IT.

Alur proses:
1. **Konfigurasi path** : menentukan lokasi file kamus
2. **Load kamus** : membangun dictionary lookup dari file CSV kamus IT
3. **Bersihkan noise** : menghapus karakter khusus yang tidak relevan
4. **Standarisasi token** : mencocokkan setiap skill ke nama standar internasional
5. **Pipeline per baris** : menggabungkan semua langkah di atas
6. **Terapkan & inspeksi** : jalankan pipeline ke seluruh DataFrame

## 3.1 Konfigurasi Path File

Mendefinisikan path file kamus standarisasi dan nama kolom yang digunakan.
Kamus berisi mapping dari nama skill mentah (token original) ke nama standar internasional.

In [28]:
# Sesuaikan path file
KAMUS_FILE  = 'Kamus_IT_Bersih_Filtered.csv'   # file kamus standarisasi

# Nama kolom
SKILL_COL   = 'skill'                           # kolom skill di data utama
TOKEN_COL   = 'Token_Original'                  # kolom token di kamus
STANDAR_COL = 'Skill_Standard_Internasional'    # kolom standar di kamus

print('✅ Konfigurasi selesai')
print(f'   Kamus  : {KAMUS_FILE}')

✅ Konfigurasi selesai
   Kamus  : Kamus_IT_Bersih_Filtered.csv


## 3.2 Load Kamus Standarisasi

File kamus dibaca dari CSV dan divalidasi untuk memastikan kolom yang diperlukan tersedia.
Kamus mendukung format CSV maupun Excel, dengan deteksi otomatis berdasarkan ekstensi file.

In [29]:
# Load kamus standarisasi
try:
    if KAMUS_FILE.endswith(".csv"):
        kamus_df = pd.read_csv(KAMUS_FILE)
    else:
        kamus_df = pd.read_excel(KAMUS_FILE)
except FileNotFoundError:
    print(f"File '{KAMUS_FILE}' tidak ditemukan. Pastikan file ada di direktori yang benar.")
    raise

# Validasi kolom kamus
assert TOKEN_COL in kamus_df.columns,   f"Kolom '{TOKEN_COL}' tidak ditemukan di kamus!"
assert STANDAR_COL in kamus_df.columns, f"Kolom '{STANDAR_COL}' tidak ditemukan di kamus!"

print(f'✅ Kamus dimuat : {len(kamus_df):,} entri')
print('\n Preview kamus (5 baris pertama):')
display(kamus_df.head())

✅ Kamus dimuat : 36,468 entri

 Preview kamus (5 baris pertama):


,Token_Original,Skill_Standard_Internasional
0,& PowerPoint,& PowerPoint
1,& deploy tools,& Deploy Tools
2,(5461) Data Analyst,(5461) Data Analyst
3,(5602) Data Engineer,(5602) Data Engineer
4,(Electronic Data Interchange) Data Analyst - Contract,(Electronic Data Interchange) Data Analyst - Contract


## 3.3 Bangun Lookup Dictionary dari Kamus

Fungsi `build_lookup()` mengkonversi DataFrame kamus menjadi dictionary Python
dengan key berupa token lowercase untuk pencocokan case-insensitive.
Dictionary memiliki kompleksitas akses O(1) sehingga sangat efisien untuk lookup massal.

In [30]:
def build_lookup(kamus_df: pd.DataFrame, token_col: str, standar_col: str) -> dict:
    """
    Bangun dictionary lookup dari kamus standarisasi.
    Returns: { 'token_lowercase': 'Skill Standard' }
    """
    kamus_df = kamus_df.copy()
    kamus_df[token_col]   = kamus_df[token_col].astype(str).str.strip()
    kamus_df[standar_col] = kamus_df[standar_col].astype(str).str.strip()
    return dict(zip(kamus_df[token_col].str.lower(), kamus_df[standar_col]))


lookup = build_lookup(kamus_df, TOKEN_COL, STANDAR_COL)

print(f'✅ Lookup dictionary dibuat: {len(lookup):,} entri')
print('\n Contoh 5 entri pertama:')
for k, v in list(lookup.items())[:5]:
    print(f'   "{k}" → "{v}"')

✅ Lookup dictionary dibuat: 36,468 entri

 Contoh 5 entri pertama:
   "& powerpoint" → "& PowerPoint"
   "& deploy tools" → "& Deploy Tools"
   "(5461) data analyst" → "(5461) Data Analyst"
   "(5602) data engineer" → "(5602) Data Engineer"
   "(electronic data interchange) data analyst - contract" → "(Electronic Data Interchange) Data Analyst - Contract"


## 3.4 Fungsi: Bersihkan Karakter Noise

Fungsi `clean_noise()` menghapus karakter khusus yang tidak relevan dari teks skill
menggunakan regex yang dikompilasi sekali di awal untuk efisiensi.

Karakter yang dihapus: `* " ' ( ) [ ] { } < > # @ ! ? ~ ` | \ ^ _ + = ; %`

Normalisasi tambahan:
- Tanda baca ganda berurutan (`..`, `--`, `//`) >> satu karakter
- Spasi atau tab berlebih → satu spasi

In [31]:
# Pola regex yang dikompilasi sekali untuk efisiensi
_NOISE_CHARS = re.compile(r'[*"\' ()\[\]{}<>#@!?~`|\\^_+=;%]')
_MULTI_PUNCT = re.compile(r'([.\-/])\1+')   # --, .., // dst → satu karakter
_MULTI_SPACE = re.compile(r'\s{2,}')         # spasi/tab ganda → satu spasi


def clean_noise(text: str) -> str:
    """Hapus karakter noise dan normalkan spasi pada string skill."""
    if not isinstance(text, str):
        return ''
    text = _NOISE_CHARS.sub(' ', text)
    text = _MULTI_PUNCT.sub(r'\1', text)
    text = _MULTI_SPACE.sub(' ', text)
    return text.strip()


# Unit test cepat
test_cases = [
    ('**Python**',        'Python'),
    ('"Machine Learning"','Machine Learning'),
    ('(React.js)',        'React.js'),
    ('#Docker',           'Docker'),
    ('C++',               'C++'),
    ('Node..js',          'Node.js'),
    ('AWS  --  EC2',      'AWS - EC2'),
]

print('🧪 Unit Test clean_noise():')
all_pass = True
for raw, expected in test_cases:
    result = clean_noise(raw)
    status = '✅' if result == expected else '❌'
    if result != expected:
        all_pass = False
    print(f'   {status}  Input: {repr(raw):<25} → Hasil: {repr(result):<25}  (Exp: {repr(expected)})')

print()
print('✅ Semua test lulus!' if all_pass else '❌ Ada test yang gagal, periksa pola regex!')

🧪 Unit Test clean_noise():
   ✅  Input: '**Python**'              → Hasil: 'Python'                   (Exp: 'Python')
   ✅  Input: '"Machine Learning"'      → Hasil: 'Machine Learning'         (Exp: 'Machine Learning')
   ✅  Input: '(React.js)'              → Hasil: 'React.js'                 (Exp: 'React.js')
   ✅  Input: '#Docker'                 → Hasil: 'Docker'                   (Exp: 'Docker')
   ❌  Input: 'C++'                     → Hasil: 'C'                        (Exp: 'C++')
   ✅  Input: 'Node..js'                → Hasil: 'Node.js'                  (Exp: 'Node.js')
   ✅  Input: 'AWS  --  EC2'            → Hasil: 'AWS - EC2'                (Exp: 'AWS - EC2')

❌ Ada test yang gagal, periksa pola regex!


## 3.5 Fungsi: Standarisasi Satu Token

Fungsi `standardize_token()` mencocokkan satu token skill ke kamus secara case-insensitive.
Jika token ditemukan di kamus, dikembalikan nama standar internasionalnya.
Jika tidak ditemukan, token asli dikembalikan tanpa perubahan
sehingga tidak ada data yang hilang.

In [32]:
# Cocokkan satu token dengan kamus standarisasi (case-insensitive). 
# Jika tidak ditemukan, kembalikan token asli.
def standardize_token(token: str, lookup: dict) -> str:
    
    token = token.strip()
    if not token:
        return ''
    return lookup.get(token.lower(), token)

# Unit test cepat
print('🧪 Unit Test standardize_token():')
sample_tokens = list(lookup.items())[:5]
for original_lower, expected_std in sample_tokens:
    result = standardize_token(original_lower, lookup)
    status = '✅' if result == expected_std else '❌'
    print(f'   {status}  Token: {repr(original_lower):<30} → Standar: {repr(result)}')

unknown = 'SkillTidakAda'
result  = standardize_token(unknown, lookup)
status  = '✅' if result == unknown else '❌'
print(f'   {status}  Token tidak ada di kamus: {repr(unknown)} → {repr(result)} (harus tetap sama)')

🧪 Unit Test standardize_token():
   ✅  Token: '& powerpoint'                 → Standar: '& PowerPoint'
   ✅  Token: '& deploy tools'               → Standar: '& Deploy Tools'
   ✅  Token: '(5461) data analyst'          → Standar: '(5461) Data Analyst'
   ✅  Token: '(5602) data engineer'         → Standar: '(5602) Data Engineer'
   ✅  Token: '(electronic data interchange) data analyst - contract' → Standar: '(Electronic Data Interchange) Data Analyst - Contract'
   ✅  Token tidak ada di kamus: 'SkillTidakAda' → 'SkillTidakAda' (harus tetap sama)


## 3.6 Fungsi: Pipeline Utama per Baris Skill

Fungsi `process_skill()` menjalankan pipeline lengkap untuk satu nilai kolom skill:

1. Tangani nilai kosong / NaN
2. Bersihkan karakter noise via `clean_noise()`
3. Tokenisasi berdasarkan koma
4. Standarisasi setiap token via `standardize_token()`
5. Hapus token kosong & duplikat (pertahankan urutan)
6. Gabung kembali dengan `', '`

In [33]:
# Pipeline lengkap: bersihkan & standarisasi satu nilai kolom skill.
def process_skill(raw_skill, lookup: dict) -> str:

    # (1) Tangani NaN / kosong
    if pd.isna(raw_skill) or str(raw_skill).strip() == '':
        return ''
    
    # (2) Bersihkan noise
    cleaned = clean_noise(str(raw_skill))
   
   # (3) Tokenisasi berdasarkan koma
    tokens = [t.strip() for t in cleaned.split(',') if t.strip()]
    
    # (4) Standarisasi setiap token
    standardized = [standardize_token(t, lookup) for t in tokens]
   
    # (5) Hapus token kosong & duplikat, pertahankan urutan
    seen   = set()
    result = []
    for s in standardized:
        if s and s not in seen:
            seen.add(s)
            result.append(s)
    
    # (6) Gabung
    return ', '.join(result)


# Uji manual
print('🧪 Uji manual process_skill():')
manual_tests = [
    '**Python**, (SQL)',
    '"Machine Learning", #Docker',
    'AWS, AWS, Azure',
    '',
    None,
]

for t in manual_tests:
    print(f'   Input  : {repr(t)}')
    print(f'   Output : {repr(process_skill(t, lookup))}')
    print()

🧪 Uji manual process_skill():
   Input  : '**Python**, (SQL)'
   Output : 'Python, SQL'

   Input  : '"Machine Learning", #Docker'
   Output : 'Machine Learning, Docker'

   Input  : 'AWS, AWS, Azure'
   Output : 'AWS, Microsoft Azure'

   Input  : ''
   Output : ''

   Input  : None
   Output : ''



## 3.7 Terapkan Pipeline ke Seluruh DataFrame

Pipeline skill diterapkan ke seluruh kolom `skill` menggunakan `apply()`.
Kolom `skill_original` disimpan sebagai backup sebelum nilai ditimpa,
kemudian dihitung jumlah baris yang mengalami perubahan.

In [34]:
# Simpan nilai asli sebagai kolom backup
df_ai['skill_original'] = df_ai[SKILL_COL].copy()

# Terapkan pipeline ke kolom skill
df_ai[SKILL_COL] = df_ai[SKILL_COL].apply(lambda x: process_skill(x, lookup))

# Hitung berapa baris yang berubah
changed = (df_ai['skill_original'].fillna('') != df_ai[SKILL_COL].fillna('')).sum()

print(f'✅ Proses selesai!')
print(f'   Total baris       : {len(df_ai):,}')
print(f'   Baris diubah      : {changed:,}')
print(f'   Baris tidak diubah: {len(df_ai) - changed:,}')

✅ Proses selesai!
   Total baris       : 11,027
   Baris diubah      : 9,834
   Baris tidak diubah: 1,193


## 3.8 Inspeksi Hasil Perubahan

Menampilkan 20 baris yang mengalami perubahan untuk verifikasi visual,
serta distribusi jumlah skill per baris setelah cleaning selesai.

In [35]:
# Tampilkan baris yang mengalami perubahan
mask_changed = df_ai['skill_original'].fillna('') != df_ai[SKILL_COL].fillna('')
df_changed   = df_ai.loc[mask_changed, ['skill_original', SKILL_COL]].copy()
df_changed.columns = ['Sebelum', 'Sesudah']

print(f'📋 Baris yang berubah ({len(df_changed):,} baris):')
display(df_changed.head(20))

# Distribusi jumlah skill per baris
df_ai['jumlah_skill'] = df_ai[SKILL_COL].apply(
    lambda x: len([s for s in str(x).split(',') if s.strip()]) if x else 0
)

print('\n📊 Distribusi jumlah skill per baris (sesudah):')
display(df_ai['jumlah_skill'].value_counts().sort_index().rename('Jumlah Baris').to_frame())

📋 Baris yang berubah (9,834 baris):


,Sebelum,Sesudah
1,"SQL, Excel","SQL, Microsoft Excel"
3,"AWS, Azure","AWS, Microsoft Azure"
10,"SQL, Oracle, Database Management","SQL, Oracle Database, Database Management"
17,"AWS, Azure, Architecture","AWS, Microsoft Azure, Architecture"
18,"Consulting, IT Strategy","IT Consulting, IT Strategy"
20,"Business Analysis, Requirements","Business Analysis, Requirements Gathering"
26,"Procurement, Vendor Management","IT Procurement, Vendor Management"
30,"AWS, Azure, Cloud Services","AWS, Microsoft Azure, Cloud Services"
34,"QA, Testing","Quality Assurance, Software Testing"
35,"iOS, Android, Mobile Development","iOS Development, Android Development, Mobile Development"



📊 Distribusi jumlah skill per baris (sesudah):


,Jumlah Baris
jumlah_skill,
0,2
1,156
2,168
3,415
4,706
...,...
78,2
79,1
87,1


---
# BAGIAN 4 — Export Final Dataset

Semua proses cleaning telah selesai. File output disimpan di sini **satu kali di akhir pipeline**.

| File | Konten |
|------|--------|
| `data_clean_Scraping_FD.xlsx` / `.csv` | Dataset scraping (Full Data) yang sudah bersih |
| `data_clean_AI_final.xlsx` / `.csv` | Dataset AI final setelah cleaning kualifikasi & skill |

Kolom bantu (`jumlah_skill`, `skill_original`, `kualifikasi_asli`) dihapus dari output final.

In [36]:
# Export Full Data (FD)
output_fd_xlsx = "data_clean_Scraping_FD.xlsx"
output_fd_csv  = "data_clean_Scraping_FD.csv"

df_scraping.to_excel(output_fd_xlsx, index=False)
df_scraping.to_csv(output_fd_csv, index=False, encoding='utf-8-sig')

print(f"✅ Full Data (FD) disimpan ke Excel : {output_fd_xlsx}")
print(f"✅ Full Data (FD) disimpan ke CSV   : {output_fd_csv}")
print(f"   Total baris FD : {df_scraping.shape[0]}, Kolom: {df_scraping.shape[1]}")

print()

# Hapus kolom bantu sebelum export AI
cols_to_drop = [c for c in ['jumlah_skill', 'skill_original', 'kualifikasi_asli'] if c in df_ai.columns]
df_ai_output = df_ai.drop(columns=cols_to_drop)

# Export Data AI Final
output_ai_xlsx = "data_clean_AI_final.xlsx"
output_ai_csv  = "data_clean_AI_final.csv"

df_ai_output.to_excel(output_ai_xlsx, index=False)
df_ai_output.to_csv(output_ai_csv, index=False, encoding='utf-8-sig')

print(f"✅ Data AI Final disimpan ke Excel : {output_ai_xlsx}")
print(f"✅ Data AI Final disimpan ke CSV   : {output_ai_csv}")
print(f"   Total baris AI : {df_ai_output.shape[0]}, Kolom: {df_ai_output.shape[1]}")

print(f"\n{'='*60}")
print("RINGKASAN AKHIR PIPELINE")
print(f"{'='*60}")
print(f"  Full Data (FD) : {df_scraping.shape[0]:,} baris, {df_scraping.shape[1]} kolom")
print(f"  Data AI Final  : {df_ai_output.shape[0]:,} baris, {df_ai_output.shape[1]} kolom")
print(f"{'='*60}")

print("\n📋 Preview Data AI Final (10 baris pertama):")
display(df_ai_output.head(10))

✅ Full Data (FD) disimpan ke Excel : data_clean_Scraping_FD.xlsx
✅ Full Data (FD) disimpan ke CSV   : data_clean_Scraping_FD.csv
   Total baris FD : 565, Kolom: 14

✅ Data AI Final disimpan ke Excel : data_clean_AI_final.xlsx
✅ Data AI Final disimpan ke CSV   : data_clean_AI_final.csv
   Total baris AI : 11027, Kolom: 7

RINGKASAN AKHIR PIPELINE
  Full Data (FD) : 565 baris, 14 kolom
  Data AI Final  : 11,027 baris, 7 kolom

📋 Preview Data AI Final (10 baris pertama):


,posisi,pendidikan,pengalaman,gender,usia,skill,kualifikasi
0,Software Engineer,terbuka untuk semua jenjang dan jurusan,0-1 tahun,tanpa ketentuan,tanpa batasan usia,"Java, Python","Menguasai Java dan Python, mampu mengembangkan software dengan pengalaman 0-1 tahun di bidang pengembangan perangkat lunak, bertanggung jawab untuk mengembangkan software."
1,Data Analyst,terbuka untuk semua jenjang dan jurusan,1-3 tahun,tanpa ketentuan,tanpa batasan usia,"SQL, Microsoft Excel","Menguasai SQL dan Excel, berpengalaman 1-3 tahun dalam bidang analisis data, bertanggung jawab untuk menganalisis data dan menyajikannya dalam bentuk yang efektif."
2,Network Engineer,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"Cisco, WAN","Menguasai teknologi Cisco dan WAN, berpengalaman dalam bidang jaringan selama 3-5 tahun, bertanggung jawab untuk menjaga dan memelihara jaringan, serta mampu melakukan perawatan dan perbaikan jaringan."
3,Cloud Architect,terbuka untuk semua jenjang dan jurusan,5+ tahun,tanpa ketentuan,tanpa batasan usia,"AWS, Microsoft Azure","Menguasai teknologi cloud seperti AWS dan Azure, berpengalaman lebih dari 5 tahun dalam bidang desain cloud, bertanggung jawab untuk merancang dan mengimplementasikan solusi cloud yang efektif dan efisien."
4,Cybersecurity Analyst,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,Cybersecurity,"Menguasai Cybersecurity, berpengalaman 3-5 tahun di bidang keamanan siber, dan bertanggung jawab untuk melindungi data."
5,IT Project Manager,terbuka untuk semua jenjang dan jurusan,5+ tahun,tanpa ketentuan,tanpa batasan usia,"PMP, Agile","Menguasai PMP dan Agile, berpengalaman lebih dari 5 tahun dalam bidang IT, bertanggung jawab untuk mengelola proyek dan mampu mengatur tim untuk mencapai tujuan proyek."
6,Data Scientist,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"R, Python","Menguasai R dan Python, berpengalaman 3-5 tahun dalam bidang analisis data, bertanggung jawab untuk menganalisis big data dan mengembangkan solusi yang efektif."
7,DevOps Engineer,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"Docker, Jenkins","Menguasai Docker dan Jenkins, berpengalaman 3-5 tahun dalam bidang DevOps, bertanggung jawab untuk mengintegrasikan dan memantau proses pengembangan dan pengelolaan infrastruktur."
8,IT Support Analyst,terbuka untuk semua jenjang dan jurusan,0-1 tahun,tanpa ketentuan,tanpa batasan usia,"Windows, Linux","Menguasai Windows dan Linux, berpengalaman dalam bidang IT dengan pengalaman kerja 0-1 tahun, bertanggung jawab untuk memberikan dukungan teknis yang efektif sebagai IT Support Analyst."
9,UX/UI Designer,terbuka untuk semua jenjang dan jurusan,1-3 tahun,tanpa ketentuan,tanpa batasan usia,UI/UX Design,"UX/UI Designer yang berpengalaman menguasai UI/UX Design, mampu mengembangkan desain user interfaces yang efektif dan efisien, serta bertanggung jawab dalam mengelola proyek desain dengan lama pengalaman 1-3 tahun di bidang desain."
